In [ ]:
# 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 기본 라이브러리
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/OnSafe/smoothing_SGV.csv")

## 정규화


*   골반의 중간점으로 중앙(0,0) 정렬
*   골반 넓이 기준으로 스케일 정규화 -> 사람의 크기 차이 보전



In [ ]:
# 중앙 정렬(골반을 기준으로 (0,0))
# 임시로 골반 인덱스 지정 : 좌골반=kp_23, 우골반=kp_24
# 행 단위로 연산이 진행됩니다.
def centralize_kp(df, pelvis_idx=(23,24)):
    """
    골반 중심으로 좌표 중앙 정렬
    pelvis_idx: tuple(좌골반, 우골반) joint index
    """

    # DataFrame 복사
    df_central = df.copy()

    # 좌골반 + 우골반 x&y 좌표 평균 -> 골반 중심 좌표 계산
    pelvis_x = (df[f'kp{pelvis_idx[0]}_x'] + df[f'kp{pelvis_idx[1]}_x']) / 2
    pelvis_y = (df[f'kp{pelvis_idx[0]}_y'] + df[f'kp{pelvis_idx[1]}_y']) / 2
    pelvis_z = (df[f'kp{pelvis_idx[0]}_z'] + df[f'kp{pelvis_idx[1]}_z']) / 2

    # x&y 좌표 열만 추출
    kp_x_cols = [c for c in df.columns if '_x' in c]
    kp_y_cols = [c for c in df.columns if '_y' in c]
    kp_z_cols = [c for c in df.columns if '_z' in c]

    # 중앙 정렬 (골반이 (0,0,0) 위치하도록 )
    for x_col, y_col, z_col in zip(kp_x_cols, kp_y_cols, kp_z_cols):
        df_central[x_col] = df_central[x_col] - pelvis_x
        df_central[y_col] = df_central[y_col] - pelvis_y
        df_central[z_col] = df_central[z_col] - pelvis_z

    return df_central


In [ ]:
# 스케일 정규화
# 키로 진행할 시 앉을 때, 서있을때, 누울 때 등 다양해서 변동하지 않는 한 관졀 골라서 스케일링 진행 : 골반
# 행 단위로 연산이 진행됩니다.
def scale_normalize_kp(df, ref_joints=(23,24)):
    """
    기준 관절 거리 기반 스케일 정규화 (예: 좌우 골반 거리)
    ref_joints: (왼쪽, 오른쪽) 기준 관절 index 튜플
    """
    df_scaled = df.copy()
    kp_x_cols = [c for c in df.columns if '_x' in c]
    kp_y_cols = [c for c in df.columns if '_y' in c]
    kp_z_cols = [c for c in df.columns if '_z' in c]

    left_x = df_scaled[f'kp{ref_joints[0]}_x']
    left_y = df_scaled[f'kp{ref_joints[0]}_y']
    left_z = df_scaled[f'kp{ref_joints[0]}_z']
    right_x = df_scaled[f'kp{ref_joints[1]}_x']
    right_y = df_scaled[f'kp{ref_joints[1]}_y']
    right_z = df_scaled[f'kp{ref_joints[1]}_z']

    # 좌우 기준 관절 거리 계산
    scale = np.sqrt((left_x - right_x)**2 + (left_y - right_y)**2 + (left_z - right_z)**2)
    scale[scale == 0] = 1

    for x_col, y_col, z_col in zip(kp_x_cols, kp_y_cols, kp_z_cols):
        df_scaled[x_col] /= scale
        df_scaled[y_col] /= scale
        df_scaled[z_col] /= scale

    return df_scaled


In [ ]:
df

,video,file_id,frame,timestamp,kp0_visibility,kp0_x,kp0_y,kp0_z,kp1_visibility,kp1_x,...,kp30_y,kp30_z,kp31_visibility,kp31_x,kp31_y,kp31_z,kp32_visibility,kp32_x,kp32_y,kp32_z
0,ADL,ADL_258_clip_06,0,0.000000,0.998187,0.786727,0.782283,-0.027047,0.998785,0.784028,...,0.515161,0.146163,0.773355,0.699561,0.457075,0.051030,0.392821,0.699123,0.503684,0.179106
1,ADL,ADL_258_clip_06,1,0.016667,0.999278,0.771839,0.793141,-0.112972,0.999355,0.770663,...,0.543091,0.146046,0.331871,0.711131,0.494385,0.127513,0.325424,0.713615,0.542298,0.199701
2,ADL,ADL_258_clip_06,2,0.033333,0.999675,0.760093,0.801706,-0.165580,0.999730,0.760093,...,0.568282,0.145136,0.556027,0.723721,0.530585,0.175476,0.328392,0.724269,0.576943,0.206547
3,ADL,ADL_258_clip_06,3,0.050000,0.999524,0.751489,0.807977,-0.184871,0.999563,0.752319,...,0.590734,0.143434,0.308033,0.737328,0.565676,0.194918,0.376794,0.731083,0.607619,0.199643
4,ADL,ADL_258_clip_06,4,0.066667,0.999344,0.750072,0.809539,-0.154387,0.999298,0.750739,...,0.612825,0.136447,0.263840,0.753178,0.594822,0.155495,0.292943,0.731319,0.625738,0.154850
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
334611,FALL,S_D_0236_clip_00,25,0.833333,0.999983,0.434655,0.472481,-0.329160,0.999980,0.439033,...,0.728999,-0.138508,0.659698,0.463071,0.712693,0.050029,0.981766,0.336311,0.718637,-0.204536
334612,FALL,S_D_0236_clip_00,26,0.866667,0.999983,0.443294,0.488036,-0.291806,0.999970,0.447451,...,0.725772,-0.141202,0.557207,0.450091,0.701186,0.064320,0.975555,0.326832,0.699618,-0.201043
334613,FALL,S_D_0236_clip_00,27,0.900000,0.999380,0.454544,0.501700,-0.256032,0.999454,0.458672,...,0.723814,-0.128142,0.558282,0.424683,0.692213,0.015982,0.865539,0.323025,0.689371,-0.183256
334614,FALL,S_D_0236_clip_00,28,0.933333,0.999671,0.468383,0.513982,-0.215460,0.999712,0.472635,...,0.719986,-0.096790,0.798776,0.387120,0.685724,-0.083463,0.891496,0.321527,0.681984,-0.146456


In [ ]:
# 중앙 정렬 적용 (영상+파일 단위)
df_centered = (
    df
    .groupby([df['video'], df['file_id'].str[:2]], group_keys=False)  # 그룹 단위
    .apply(centralize_kp)                             # 각 그룹에 중앙 정렬 적용
    .reset_index(drop=True)                            # 인덱스 초기화
)

pd.set_option('display.max_columns', None)    # 열 모두 보기

display(df_centered.head())

/tmp/ipykernel_5986/3830562095.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(centralize_kp)                             # 각 그룹에 중앙 정렬 적용


,video,file_id,frame,timestamp,kp0_visibility,kp0_x,kp0_y,kp0_z,kp1_visibility,kp1_x,kp1_y,kp1_z,kp2_visibility,kp2_x,kp2_y,kp2_z,kp3_visibility,kp3_x,kp3_y,kp3_z,kp4_visibility,kp4_x,kp4_y,kp4_z,kp5_visibility,kp5_x,kp5_y,kp5_z,kp6_visibility,kp6_x,kp6_y,kp6_z,kp7_visibility,kp7_x,kp7_y,kp7_z,kp8_visibility,kp8_x,kp8_y,kp8_z,kp9_visibility,kp9_x,kp9_y,kp9_z,kp10_visibility,kp10_x,kp10_y,kp10_z,kp11_visibility,kp11_x,kp11_y,kp11_z,kp12_visibility,kp12_x,kp12_y,kp12_z,kp13_visibility,kp13_x,kp13_y,kp13_z,kp14_visibility,kp14_x,kp14_y,kp14_z,kp15_visibility,kp15_x,kp15_y,kp15_z,kp16_visibility,kp16_x,kp16_y,kp16_z,kp17_visibility,kp17_x,kp17_y,kp17_z,kp18_visibility,kp18_x,kp18_y,kp18_z,kp19_visibility,kp19_x,kp19_y,kp19_z,kp20_visibility,kp20_x,kp20_y,kp20_z,kp21_visibility,kp21_x,kp21_y,kp21_z,kp22_visibility,kp22_x,kp22_y,kp22_z,kp23_visibility,kp23_x,kp23_y,kp23_z,kp24_visibility,kp24_x,kp24_y,kp24_z,kp25_visibility,kp25_x,kp25_y,kp25_z,kp26_visibility,kp26_x,kp26_y,kp26_z,kp27_visibility,kp27_x,kp27_y,kp27_z,kp28_visibility,kp28_x,kp28_y,kp28_z,kp29_visibility,kp29_x,kp29_y,kp29_z,kp30_visibility,kp30_x,kp30_y,kp30_z,kp31_visibility,kp31_x,kp31_y,kp31_z,kp32_visibility,kp32_x,kp32_y,kp32_z
0,ADL,ADL_258_clip_06,0,0.000000,0.998187,0.023899,0.240291,-0.027074,0.998785,0.021201,0.250606,-0.040121,0.998666,0.020745,0.250346,-0.040149,0.999087,0.020230,0.250176,-0.040153,0.997931,0.020708,0.251502,-0.025606,0.997484,0.019840,0.251742,-0.025623,0.998213,0.019146,0.251956,-0.025548,0.998898,0.014905,0.245438,-0.073013,0.998149,0.013126,0.248408,-0.006877,0.997672,0.022469,0.228917,-0.036199,0.996905,0.021552,0.230007,-0.017354,0.999817,0.012135,0.192680,-0.104855,0.999056,0.004927,0.203456,0.033423,0.970604,0.002443,0.175573,-0.144971,0.368704,-0.000906,0.225685,0.042415,0.941348,0.021929,0.251500,-0.158791,0.456000,0.005594,0.306600,0.030730,0.910983,0.027863,0.272471,-0.182633,0.426268,0.005700,0.332626,0.024729,0.906079,0.028285,0.279831,-0.170811,0.421793,0.008888,0.334210,0.017493,0.882867,0.026691,0.271332,-0.155078,0.402013,0.009235,0.324136,0.025416,0.999299,0.000587,-0.003660,-0.044557,0.998805,-0.000587,0.003660,0.044557,0.784091,-0.001778,-0.024593,-0.063891,0.237650,-0.030295,0.032618,0.064200,0.734530,-0.055028,-0.077610,0.019132,0.265583,-0.038792,0.025304,0.169155,0.751315,-0.047609,-0.040585,0.038915,0.322398,-0.050254,-0.026830,0.146136,0.773355,-0.063267,-0.084917,0.051003,0.392821,-0.063705,-0.038308,0.179079
1,ADL,ADL_258_clip_06,1,0.016667,0.999278,0.026849,0.241283,-0.113034,0.999355,0.025673,0.249492,-0.127918,0.999208,0.025696,0.248667,-0.127960,0.999381,0.025649,0.247949,-0.127944,0.999230,0.023781,0.251801,-0.114718,0.999156,0.022494,0.252480,-0.114774,0.999339,0.021386,0.253140,-0.114715,0.998766,0.020860,0.241293,-0.158165,0.999073,0.014581,0.248766,-0.097998,0.999270,0.025819,0.228860,-0.119943,0.999378,0.023678,0.231138,-0.102591,0.999865,0.020295,0.181653,-0.175189,0.999810,-0.001237,0.206897,-0.050200,0.971459,0.025588,0.156144,-0.216299,0.826564,0.003169,0.208705,-0.047837,0.971444,0.049422,0.231375,-0.247499,0.893059,0.012558,0.285260,-0.096312,0.954860,0.055960,0.249668,-0.280543,0.845720,0.012501,0.309893,-0.110698,0.951892,0.055589,0.259758,-0.272347,0.839212,0.014923,0.313793,-0.120161,0.931389,0.052870,0.251104,-0.246066,0.811819,0.015766,0.303318,-0.102335,0.999314,0.003871,-0.007843,-0.041000,0.999208,-0.003871,0.007843,0.041000,0.600765,0.012716,-0.040335,-0.019861,0.207938,-0.006568,0.029439,0.056696,0.296731,-0.030402,-0.073867,0.098510,0.299585,-0.014217,0.023248,0.155472,0.335199,-0.025701,-0.041246,0.117413,0.260233,-0.021049,-0.008768,0.145983,0.331871,-0.033859,-0.057474,0.127451,0.325424,-0.031375,-0.009561,0.199639
2,ADL,ADL_258_clip_06,2,0.033333,0.999675,0.028789,0.242917,-0.165660,0.999730,0.028789,0.249769,-0.181936,0.999652,0.029197,0.248637,-0.181984,0.999715,0.029531,0.247620,-0.181949,0.999645,0.025833,0.252849,-0.170199,0.999583,0.024222,0.253765,-0.170278,0.999681,0.022

In [ ]:
# 스케일링 적용 예시
df_scaled = (
    df_centered
    .groupby([df['video'], df['file_id'].str[:2]], group_keys=False)
    .apply(scale_normalize_kp)
    .reset_index(drop=True)
)

# 결과 확인
display(df_scaled.head())

,video,file_id,frame,timestamp,kp0_visibility,kp0_x,kp0_y,kp0_z,kp1_visibility,kp1_x,kp1_y,kp1_z,kp2_visibility,kp2_x,kp2_y,kp2_z,kp3_visibility,kp3_x,kp3_y,kp3_z,kp4_visibility,kp4_x,kp4_y,kp4_z,kp5_visibility,kp5_x,kp5_y,kp5_z,kp6_visibility,kp6_x,kp6_y,kp6_z,kp7_visibility,kp7_x,kp7_y,kp7_z,kp8_visibility,kp8_x,kp8_y,kp8_z,kp9_visibility,kp9_x,kp9_y,kp9_z,kp10_visibility,kp10_x,kp10_y,kp10_z,kp11_visibility,kp11_x,kp11_y,kp11_z,kp12_visibility,kp12_x,kp12_y,kp12_z,kp13_visibility,kp13_x,kp13_y,kp13_z,kp14_visibility,kp14_x,kp14_y,kp14_z,kp15_visibility,kp15_x,kp15_y,kp15_z,kp16_visibility,kp16_x,kp16_y,kp16_z,kp17_visibility,kp17_x,kp17_y,kp17_z,kp18_visibility,kp18_x,kp18_y,kp18_z,kp19_visibility,kp19_x,kp19_y,kp19_z,kp20_visibility,kp20_x,kp20_y,kp20_z,kp21_visibility,kp21_x,kp21_y,kp21_z,kp22_visibility,kp22_x,kp22_y,kp22_z,kp23_visibility,kp23_x,kp23_y,kp23_z,kp24_visibility,kp24_x,kp24_y,kp24_z,kp25_visibility,kp25_x,kp25_y,kp25_z,kp26_visibility,kp26_x,kp26_y,kp26_z,kp27_visibility,kp27_x,kp27_y,kp27_z,kp28_visibility,kp28_x,kp28_y,kp28_z,kp29_visibility,kp29_x,kp29_y,kp29_z,kp30_visibility,kp30_x,kp30_y,kp30_z,kp31_visibility,kp31_x,kp31_y,kp31_z,kp32_visibility,kp32_x,kp32_y,kp32_z
0,ADL,ADL_258_clip_06,0,0.000000,0.998187,0.267265,2.687150,-0.302766,0.998785,0.237088,2.802501,-0.448672,0.998666,0.231984,2.799595,-0.448985,0.999087,0.226229,2.797692,-0.449021,0.997931,0.231579,2.812513,-0.286344,0.997484,0.221866,2.815207,-0.286541,0.998213,0.214106,2.817597,-0.285702,0.998898,0.166676,2.744710,-0.816492,0.998149,0.146782,2.777923,-0.076903,0.997672,0.251268,2.559947,-0.404811,0.996905,0.241018,2.572137,-0.194066,0.999817,0.135710,2.154723,-1.172584,0.999056,0.055099,2.275222,0.373767,0.970604,0.027319,1.963408,-1.621192,0.368704,-0.010134,2.523808,0.474321,0.941348,0.245233,2.812492,-1.775739,0.456000,0.062563,3.428669,0.343652,0.910983,0.311587,3.047014,-2.042369,0.426268,0.063737,3.719719,0.276539,0.906079,0.316304,3.129313,-1.910160,0.421793,0.099394,3.737430,0.195624,0.882867,0.298478,3.034279,-1.734217,0.402013,0.103271,3.624776,0.284221,0.999299,0.006567,-0.040929,-0.498279,0.998805,-0.006567,0.040929,0.498279,0.784091,-0.019878,-0.275024,-0.714485,0.237650,-0.338790,0.364763,0.717940,0.734530,-0.615373,-0.867901,0.213950,0.265583,-0.433803,0.282968,1.891646,0.751315,-0.532409,-0.453859,0.435186,0.322398,-0.561983,-0.300039,1.634220,0.773355,-0.707502,-0.949614,0.570357,0.392821,-0.712404,-0.428392,2.002616
1,ADL,ADL_258_clip_06,1,0.016667,0.999278,0.320220,2.877692,-1.348117,0.999355,0.306190,2.975596,-1.525627,0.999208,0.306466,2.965760,-1.526133,0.999381,0.305909,2.957200,-1.525937,0.999230,0.283633,3.003134,-1.368202,0.999156,0.268282,3.011231,-1.368867,0.999339,0.255060,3.019111,-1.368160,0.998766,0.248787,2.877814,-1.886380,0.999073,0.173908,2.966939,-1.168790,0.999270,0.307931,2.729528,-1.430514,0.999378,0.282403,2.756704,-1.223568,0.999865,0.242046,2.166507,-2.089415,0.999810,-0.014754,2.467583,-0.598718,0.971459,0.305178,1.862267,-2.579717,0.826564,0.037801,2.489146,-0.570530,0.971444,0.589437,2.759524,-2.951833,0.893059,0.149769,3.402194,-1.148683,0.954860,0.667411,2.977698,-3.345931,0.845720,0.149092,3.695974,-1.320250,0.951892,0.662987,3.098035,-3.248181,0.839212,0.177977,3.742488,-1.433110,0.931389,0.630556,2.994825,-2.934738,0.811819,0.188033,3.617563,-1.220508,0.999314,0.046173,-0.093539,-0.488997,0.999208,-0.046173,0.093539,0.488997,0.600765,0.151654,-0.481059,-0.236871,0.207938,-0.078338,0.351105,0.676197,0.296731,-0.362599,-0.880980,1.174892,0.299585,-0.169558,0.277265,1.854257,0.335199,-0.306524,-0.491922,1.400337,0.260233,-0.251046,-0.104569,1.741090,0.331871,-0.403819,-0.685467,1.520057,0.325424,-0.374196,-0.114031,2.381021
2,ADL,ADL_258_clip_06,2,0.033333,0.999675,0.364484,3.075438,-2.097327,0.999730,0.364482,3.162179,-2.303382,0.999652,0.369649,3.147843,-2.303998,0.999715,0.373877,3.134975,-2.303549,0.999645,0.327060,3.201169,-2.154788,0.999583,0.306655,3.212773,-2.155791,0.999681,0.288

In [ ]:
df_scaled.to_csv("/content/drive/MyDrive/OnSafe/scaled_df.csv", index=False, encoding="utf-8")